In [ ]:
import os
import subprocess
import sys
import glob
import re

# NLGCL Kaggle Runner v12 (Fix: Manual 5-core filtering in preprocessing and cache cleanup)
# 1. Clone the repository if not present
if not os.path.exists('main.py'):
    !git clone https://github.com/yangzeha/NLGCL.git
    %cd NLGCL

# 2. Patch Code for Numpy 2.0 Compatibility
print("Patching RecBole for Numpy 2.x compatibility...")
files = glob.glob('recbole/**/*.py', recursive=True) + glob.glob('recbole_gnn/**/*.py', recursive=True)

# Use Regex to avoid partial matches
replacements = [
    (r'np\.float\b', 'float'),
    (r'np\.int\b', 'int'),
    (r'np\.object\b', 'object'),
    (r'np\.bool\b', 'bool'),
    (r'np\.str\b', 'str'),
    (r'np\.long\b', 'int'),
]

patched_count = 0
for file_path in files:
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()
        
        new_content = content
        
        # 1. Disable generic compatibility_settings to avoid crashes
        if 'def compatibility_settings(self):' in new_content:
            new_content = re.sub(
                r'def compatibility_settings\(self\):.*?(?=\n\s+def|\Z)', 
                'def compatibility_settings(self):\n        pass\n\n    ', 
                new_content, 
                flags=re.DOTALL
            )

        # 2. Safe replacements for types
        for pattern, replacement in replacements:
            new_content = re.sub(pattern, replacement, new_content)
            
        if new_content != content:
            with open(file_path, 'w', encoding='utf-8') as f:
                f.write(new_content)
            patched_count += 1
    except Exception as e:
        print(f"Skipping {file_path}: {e}")

print(f"Successfully patched {patched_count} files.")

# 3. Setup Dependencies
print("Configuring environment...")

# Force install Numpy 2.x and Pandas 2.2.x
print("Force upgrading to Numpy 2.0+ and Pandas 2.2+...")
!pip install "numpy>=2.0.0" "pandas>=2.2.2" --force-reinstall --upgrade

print("Installing PyTorch 2.4.0...")
!pip install torch==2.4.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

print("Installing PyG wheels...")
!pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.4.0+cu121.html

requirements_content = """colorlog
tensorboard
six
colorama
pyyaml
hyperopt>=0.2.7
torch_geometric
"""
with open('requirements.txt', 'w') as f:
    f.write(requirements_content)

print("Installing other requirements...")
!pip install -r requirements.txt

# 4. Dataset Preprocessing (Isolated Script)
preprocess_script = r"""
import os
import pandas as pd
import re

inter_path = 'dataset/QB-video/QB-video.inter'
csv_path = 'dataset/QB-video.csv'
source_path = None

if os.path.exists(csv_path):
    source_path = csv_path
elif os.path.exists(inter_path):
    source_path = inter_path

def apply_k_core(df, k=5, user_col='user_id', item_col='item_id'):
    """Apply recursive k-core filtering directly in pandas to ensure data quality."""
    print(f"Applying recursive {k}-core filtering...")
    print(f"Initial shape: {df.shape}, Users: {df[user_col].nunique()}, Items: {df[item_col].nunique()}")
    
    iteration = 0
    while True:
        iteration += 1
        n_rows_before = len(df)
        
        # Filter users
        user_counts = df[user_col].value_counts()
        valid_users = user_counts[user_counts >= k].index
        df = df[df[user_col].isin(valid_users)]
        
        # Filter items
        item_counts = df[item_col].value_counts()
        valid_items = item_counts[item_counts >= k].index
        df = df[df[item_col].isin(valid_items)]
        
        n_rows_after = len(df)
        print(f"  Iteration {iteration}: {n_rows_after} interactions remaining.")
        
        if n_rows_after == n_rows_before:
            break
            
    print(f"Final shape: {df.shape}, Users: {df[user_col].nunique()}, Items: {df[item_col].nunique()}")
    return df

if source_path:
    print(f"Processing dataset from {source_path}...")
    df = pd.read_csv(source_path)
    
    # Check if headers are already RecBole format
    is_recbole_fmt = ':' in str(df.columns[0])
    
    if not is_recbole_fmt:
        print("Detected raw CSV header.")
        if 'user_id' in df.columns and 'item_id' in df.columns:
            # Drop duplicates first
            df = df.drop_duplicates(subset=['user_id', 'item_id'])
            
            # Apply 5-core filtering MANUALLY
            df = apply_k_core(df, k=5, user_col='user_id', item_col='item_id')
            
            # Rename for RecBole
            df = df[['user_id', 'item_id']]
            df.columns = ['user_id:token', 'item_id:token']
            
            os.makedirs(os.path.dirname(inter_path), exist_ok=True)
            df.to_csv(inter_path, index=False, sep=',')
            print(f"Successfully saved filtered data to {inter_path}")
        else:
            print("Error: Could not find user_id/item_id columns.")
    else:
        print("Dataset header already contains type annotations. Skipping manual CSV filtering.")
else:
    print("Warning: No dataset file found.")

# Update Config with Essential Filtering
config_path = 'properties/QB-video.yaml'
if os.path.exists(config_path):
    with open(config_path, 'r') as f:
        content = f.read()
    
    new_content = content
    
    # 1. Ensure field_separator and proper formatting
    if 'field_separator' not in new_content:
        new_content = new_content.replace('inter: [user_id, item_id]', 'inter: [user_id, item_id]\n\nfield_separator: ","')
    
    # Clean up any bad lines from previous edits
    if 'val_interval: "[0,inf)"' in new_content:
        new_content = new_content.replace('val_interval: "[0,inf)"', '')
    new_content = re.sub(r'\n\s*val_interval: "\[0,inf\)"', '', new_content)

    # Note: Since we pre-filtered manually, RecBole filtering is technically redundant but harmless.
    # We keep it as a safety net, but rely on the manual filter for correctness.
    if 'user_inter_num_interval' not in new_content:
         new_content += '\nuser_inter_num_interval: "[5,inf)"'
    if 'item_inter_num_interval' not in new_content:
         new_content += '\nitem_inter_num_interval: "[5,inf)"'
    if 'filter_inter_by_user_or_item' not in new_content:
        new_content += '\nfilter_inter_by_user_or_item: true'

    if new_content != content:
        with open(config_path, 'w') as f:
            f.write(new_content)
        print("Updated properties/QB-video.yaml config.")
"""

with open('preprocess_dataset.py', 'w') as f:
    f.write(preprocess_script)

print("Running preprocessing in isolated subprocess...")
!python preprocess_dataset.py

# 5. Clear Cache and Run Training
print("Clearing RecBole dataset cache (saved/) to enforce fresh loading...")
import shutil
if os.path.exists('saved'):
    shutil.rmtree('saved')
    print("Deleted 'saved/' directory.")

print("Starting Training...")
!python main.py --dataset QB-video